# 05_knowledge_base_and_retrieval

## Extract text

Extract clean text from each knowledge base document.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from pypdf import PdfReader

pd.set_option("display.width", 200)
KB = Path("../knowledge_base")
RAWD, PROC = KB / "raw_docs", KB / "processed"
(PROC / "texts").mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(RAWD / "MANIFEST.csv")
print("documents:", len(manifest), "| indexed by default:", int(manifest["index_by_default"].sum()))

def tidy(s):
    s = re.sub(r"[ \t\r\f\v]+", " ", s)
    return re.sub(r"\n\s*\n+", "\n\n", s).strip()

def html_text(path, selector, drop=()):
    soup = BeautifulSoup(path.read_text(encoding="utf-8", errors="ignore"), "html.parser")
    root = soup.select_one(selector) or soup.body
    for t in root(["script", "style", "nav", "footer", "form", "noscript"]):
        t.decompose()
    for sel in drop:
        for t in root.select(sel):
            t.decompose()
    blocks = []
    for el in root.find_all(["h1", "h2", "h3", "h4", "p", "li"]):
        txt = re.sub(r"\s+", " ", el.get_text(" ", strip=True))
        if txt:
            blocks.append(("## " + txt) if el.name in ("h1", "h2", "h3", "h4") else txt)
    joined = "\n\n".join(blocks)
    full = root.get_text("\n", strip=True)
    return tidy(joined if len(joined) >= 0.6 * len(full) else full)

def pdf_text(path):
    return tidy("\n\n".join((p.extract_text() or "") for p in PdfReader(str(path)).pages))

WIKI_DROP = ("table", ".reflist", ".navbox", "sup.reference", ".mw-editsection", ".hatnote", ".thumb",
             ".metadata", ".sistersitebox", "ol.references")

def extract(doc_id, path):
    if doc_id in (2, 3, 4, 5, 6, 7, 8, 14, 15, 16, 17, 18, 19):      # NOAA pages: the <article> tag
        return html_text(path, "article")
    if doc_id == 9:                                                  # GFZ
        return html_text(path, "main")
    if doc_id == 13:                                                 # Wikipedia
        t = html_text(path, "#mw-content-text", WIKI_DROP)
        cut = [m.start() for m in re.finditer(r"## (References|External links|See also)", t)]
        t = t[:min(cut)].strip() if cut else t
        return tidy("\n".join(l for l in t.split("\n") if "GLAT" not in l and "MLAT" not in l))
    if doc_id in (11, 20):                                           # NASA, Kyoto Dst
        return html_text(path, "body")
    if doc_id == 12:                                                 # the paper: stop before the ORCID list
        t = pdf_text(path)
        cut = [m.start() for m in re.finditer(r"\n\s*ORCID iDs\s*\n", t)]
        if cut:
            t = t[:cut[0]]
        return tidy(re.sub(r"(?m)^The Astrophysical Journal, 979:49.*$|^Hayakawa et al\.\s*$", "", t))
    return pdf_text(path)                                            # 1 and 21

rows = []
for r in manifest[manifest["index_by_default"]].itertuples():
    text = extract(r.doc_id, RAWD / r.file)
    (PROC / "texts" / f"{r.doc_id:02d}.txt").write_text(text, encoding="utf-8")
    rows.append({"doc_id": r.doc_id, "role": r.role, "chars": len(text), "words": len(text.split())})
stats = pd.DataFrame(rows)
print()
print(stats.to_string(index=False))
print("\ntotal characters:", stats["chars"].sum())
print("\n--- doc 3, first 400 characters ---\n" + (PROC / "texts" / "03.txt").read_text(encoding="utf-8")[:400])
print("\n--- doc 12, last 250 characters ---\n" + (PROC / "texts" / "12.txt").read_text(encoding="utf-8")[-250:])

documents: 21 | indexed by default: 20

 doc_id           role  chars  words
      1     definition   8008   1191
      2     definition   5923    842
      3        physics   3300    511
      4        physics   2542    418
      5        physics   4171    654
      6        physics   3014    494
      7         impact   6135    963
      8         impact   2972    425
      9     definition   6515   1050
     11        physics   2245    363
     12 event_2024_may  90383  15103
     13 event_2024_may  10045   1659
     14         impact   2040    322
     15         impact   9664   1481
     16         impact   3042    468
     17         impact   2754    470
     18        physics   3374    541
     19     definition   2218    357
     20     definition  20872   3343
     21     definition   9651   1503

total characters: 198868

--- doc 3, first 400 characters ---
A geomagnetic storm is a major disturbance of Earth's magnetosphere that occurs when there is a very efficient exchange 

Split the texts into chunks.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def sections(text):
    """Split a document into (heading, body) pairs using the '## ' lines."""
    out, head, buf = [], "", []
    for line in text.split("\n"):
        if line.startswith("## "):
            if "".join(buf).strip():
                out.append((head, "\n".join(buf).strip()))
            head, buf = line[3:].strip(), []
        else:
            buf.append(line)
    if "".join(buf).strip():
        out.append((head, "\n".join(buf).strip()))
    return out

splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=120,
                                          separators=["\n\n", "\n", ". ", " "])
rows = []
for r in manifest[manifest["index_by_default"]].itertuples():
    text = (PROC / "texts" / f"{r.doc_id:02d}.txt").read_text(encoding="utf-8")
    k = 0
    for head, body in sections(text):
        for piece in splitter.split_text(body):
            if len(piece) < 120:                       # drop fragments too short to carry meaning
                continue
            rows.append({"chunk_id": f"d{r.doc_id:02d}_c{k:03d}", "doc_id": r.doc_id, "role": r.role,
                         "title": r.title, "section": head, "source_url": r.source_url,
                         "licence_status": r.licence_status, "text": piece})
            k += 1
chunks = pd.DataFrame(rows)
chunks.to_json(PROC / "chunks.jsonl", orient="records", lines=True)

print("chunks:", len(chunks), "| mean chars:", round(chunks["text"].str.len().mean()),
      "| min / max:", chunks["text"].str.len().min(), "/", chunks["text"].str.len().max())
print("chunks per document:", chunks.groupby("doc_id").size().to_dict())
print("chunks per role    :", chunks.groupby("role").size().to_dict())
print()
print(chunks.loc[chunks["doc_id"] == 3, ["chunk_id", "section", "text"]].head(2).to_string(index=False))

chunks: 290 | mean chars: 715 | min / max: 138 / 898
chunks per document: {1: 11, 2: 9, 3: 5, 4: 4, 5: 7, 6: 6, 7: 10, 8: 4, 9: 10, 11: 5, 12: 122, 13: 15, 14: 3, 15: 15, 16: 5, 17: 4, 18: 5, 19: 4, 20: 32, 21: 14}
chunks per role    : {'definition': 80, 'event_2024_may': 137, 'impact': 41, 'physics': 32}

chunk_id section                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

Embed the chunks, build the vector index and try two searches.

In [3]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

model = SentenceTransformer(MODEL, device="cpu")
texts = [f"{r.title}. {r.text}" for r in chunks.itertuples()]
emb = model.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=False)
print("embeddings:", emb.shape)

client = chromadb.PersistentClient(path=str(KB / "vector_store"), settings=Settings(anonymized_telemetry=False))
try:
    client.delete_collection("spaceweather_kb")
except Exception:
    pass
col = client.create_collection("spaceweather_kb", metadata={"hnsw:space": "cosine"})
col.add(ids=chunks["chunk_id"].tolist(), embeddings=emb.tolist(), documents=chunks["text"].tolist(),
        metadatas=[{"doc_id": int(r.doc_id), "role": r.role, "title": r.title, "section": r.section,
                    "source_url": r.source_url, "licence_status": r.licence_status} for r in chunks.itertuples()])
print("indexed chunks:", col.count())

def search(query, k=5, where=None):
    q = model.encode([QUERY_PREFIX + query], normalize_embeddings=True)
    res = col.query(query_embeddings=q.tolist(), n_results=k, where=where)
    return [{"chunk_id": i, "doc_id": m["doc_id"], "title": m["title"], "score": round(1 - d, 3), "text": t}
            for i, m, d, t in zip(res["ids"][0], res["metadatas"][0], res["distances"][0], res["documents"][0])]

GENERAL = {"role": {"$ne": "event_2024_may"}}          # leave out the event-specific documents

for q in ["Why does a southward interplanetary magnetic field matter for geomagnetic storms?",
          "How can geomagnetic storms affect power grids and transformers?"]:
    print("\nQ:", q)
    for r in search(q, 3, GENERAL):
        print(f"  doc {r['doc_id']:>2} | {r['score']:.3f} | {r['text'][:110]!r}")

/Users/manasstark/Projects/Spaceweather_GENAI_RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1025 19:23:37.081000 76808 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|███████████████████████| 199/199 [00:00<00:00, 4351.62it/s]


embeddings: (290, 384)
indexed chunks: 290

Q: Why does a southward interplanetary magnetic field matter for geomagnetic storms?
  doc  3 | 0.817 | "A geomagnetic storm is a major disturbance of Earth's magnetosphere that occurs when there is a very efficient"
  doc  7 | 0.794 | '. Geomagnetic storms in turn are caused by disturbances that propagate away from the Sun, travel through inter'
  doc 11 | 0.780 | "When the solar wind and the magnetosphere interact they operate as a system. The Earth's magnetosphere is full"

Q: How can geomagnetic storms affect power grids and transformers?
  doc  7 | 0.854 | 'Assessing the impact of geomagnetic storms on the electrical power grid involves a number of considerations. T'
  doc  7 | 0.852 | '. Geomagnetic storms in turn are caused by disturbances that propagate away from the Sun, travel through inter'
  doc  7 | 0.838 | '. This means that a system that is near peak levels of demand prior to the geomagnetic storm event may not be '


Check how well retrieval works on 18 hand-labelled questions.

In [4]:
labelled = [
    ("What Kp value corresponds to a severe G4 geomagnetic storm?", {1, 2}, GENERAL),
    ("What does the Kp index measure and how is it calculated?", {9}, GENERAL),
    ("What does the Dst index measure?", {20}, GENERAL),
    ("What is the difference between final, provisional and real-time Dst?", {21}, GENERAL),
    ("Why does a southward interplanetary magnetic field matter for geomagnetic storms?", {3, 18}, GENERAL),
    ("What is a coronal mass ejection and how long does it take to reach Earth?", {5}, GENERAL),
    ("What are high-speed solar wind streams from coronal holes?", {6, 3}, GENERAL),
    ("How can geomagnetic storms affect power grids and transformers?", {7}, GENERAL),
    ("How does a geomagnetic storm affect satellite drag?", {15}, GENERAL),
    ("How does space weather degrade GPS navigation?", {16}, GENERAL),
    ("Which radio communications are affected during geomagnetic storms?", {14}, GENERAL),
    ("How far south can aurora be seen during a severe storm?", {17, 1}, GENERAL),
    ("What is F10.7 and what does it indicate about solar activity?", {19}, GENERAL),
    ("How does the solar wind transfer energy into the magnetosphere?", {18, 3, 4}, GENERAL),
    ("What problems can spacecraft charging cause during extreme storms?", {1, 2}, GENERAL),  # label fixed after checking the text: docs 1 and 2 mention surface charging, doc 8 does not
    ("What is the solar wind and how fast does it normally flow?", {4}, GENERAL),
    ("What caused the May 2024 geomagnetic storm?", {12, 13}, None),
    ("What was the minimum Dst reached in the May 2024 storm?", {12, 13}, None),
]

rows = []
for q, rel, w in labelled:
    res = search(q, 5, w)
    docs = [r["doc_id"] for r in res]
    rank = next((i + 1 for i, d in enumerate(docs) if d in rel), np.nan)
    rows.append({"question": q[:60], "relevant": sorted(rel), "retrieved": docs, "rank": rank, "top1_score": res[0]["score"]})
ev_ret = pd.DataFrame(rows)
print(ev_ret.to_string(index=False))
print()
print("hit@1:", round((ev_ret["rank"] == 1).mean(), 2), "| hit@3:", round((ev_ret["rank"] <= 3).mean(), 2),
      "| hit@5:", round((ev_ret["rank"] <= 5).mean(), 2), "| MRR:", round((1 / ev_ret["rank"]).fillna(0).mean(), 3))

print("\nquestions with little or no real support (top-1 similarity and document):")
for q in ["What is the economic cost of a Carrington-class solar storm?",
          "How do geomagnetic storms affect commercial aviation radiation exposure?",
          "How does space weather affect the stock market?",
          "What is the capital of France?"]:
    r = search(q, 1, GENERAL)[0]
    print(f"  {r['score']:.3f} doc {r['doc_id']:>2} | {q}")
print("\ntop-1 similarity of the 18 real questions: min", ev_ret["top1_score"].min(),
      "| median", ev_ret["top1_score"].median(), "| max", ev_ret["top1_score"].max())

                                                    question   relevant            retrieved  rank  top1_score
 What Kp value corresponds to a severe G4 geomagnetic storm?     [1, 2]      [9, 1, 9, 3, 1]     2       0.749
    What does the Kp index measure and how is it calculated?        [9]      [9, 9, 9, 9, 9]     1       0.780
                            What does the Dst index measure?       [20] [20, 20, 20, 20, 20]     1       0.810
What is the difference between final, provisional and real-t       [21] [21, 21, 21, 21, 21]     1       0.757
Why does a southward interplanetary magnetic field matter fo    [3, 18]     [3, 7, 11, 3, 3]     1       0.817
What is a coronal mass ejection and how long does it take to        [5]      [5, 3, 5, 6, 5]     1       0.833
  What are high-speed solar wind streams from coronal holes?     [3, 6]      [4, 6, 6, 6, 6]     2       0.813
How can geomagnetic storms affect power grids and transforme        [7]      [7, 7, 7, 7, 7]     1       0.854
 

Run the retrieval for the 15 evaluation events and save the results.

In [5]:
import json

CARDS = {}
for line in open("../data/events/event_cards.jsonl", encoding="utf-8"):
    c = json.loads(line)
    c.pop("evaluation_role", None)               # the model must not see which events are in the evaluation set
    CARDS[c["event_id"]] = c
eval_ids = pd.read_csv("../evaluation/reference_events/evaluation_set.csv", index_col="event_id").index.tolist()

def overlaps_may2024(c):
    a, b = pd.Timestamp(c["window"]["start"][:-1]), pd.Timestamp(c["window"]["end"][:-1])
    return a <= pd.Timestamp("2024-05-20") and b >= pd.Timestamp("2024-05-08")

def event_queries(c):
    g = c["geomagnetic"]["noaa_g_level_approx"]
    cls = c["class"]
    if c["type"] == "storm":
        qs = {"physics": f"How southward Bz, fast solar wind and a strong electric field drive a {cls} geomagnetic storm",
              "impacts": f"Effects of a {g or 'minor'} geomagnetic storm on power systems, spacecraft, satellite navigation and HF radio",
              "indices": f"What do Kp and Dst values mean for a {cls} storm"}
    else:
        qs = {"physics": "Solar wind and magnetic field conditions during quiet geomagnetic activity",
              "indices": "What Kp and Dst values indicate quiet conditions and no storm"}
    if not c["dst"]["minimum_is_final"]:
        qs["quality"] = "Provisional versus final Dst index values"
    return qs

def retrieve_for_event(c, k=3):
    where = None if overlaps_may2024(c) else GENERAL      # May 2024 documents only for events in that window
    return {name: search(q, k, where) for name, q in event_queries(c).items()}

summary = []
for eid in eval_ids:
    c = CARDS[eid]
    res = retrieve_for_event(c)
    row = {"event": eid, "class": c["class"], "may2024_docs_allowed": overlaps_may2024(c)}
    for name, r in res.items():
        row[name] = f"{r[0]['doc_id']} ({r[0]['score']:.2f})"
    row["all_docs"] = sorted({x["doc_id"] for r in res.values() for x in r})
    summary.append(row)
summary = pd.DataFrame(summary).fillna("-")
print(summary.to_string(index=False))

leak = [r["event"] for r in summary.to_dict("records")
        if not r["may2024_docs_allowed"] and (12 in r["all_docs"] or 13 in r["all_docs"])]
print("\nevents that received May 2024 documents without being in that window:", leak)

for eid in ["E216", "E143"]:
    print(f"\n=== what retrieval returns for {eid} ({CARDS[eid]['class']}) ===")
    for name, r in retrieve_for_event(CARDS[eid], 2).items():
        print(f"  [{name}]")
        for x in r:
            print(f"     doc {x['doc_id']:>2} | {x['score']:.3f} | {x['text'][:150]!r}")

RES = Path("../evaluation/results")
RES.mkdir(parents=True, exist_ok=True)
ev_ret.assign(relevant=ev_ret["relevant"].astype(str), retrieved=ev_ret["retrieved"].astype(str)) \
      .to_csv(RES / "retrieval_questions.csv", index=False)
summary.assign(all_docs=summary["all_docs"].astype(str)).to_csv(RES / "retrieval_events.csv", index=False)
print("\nsaved evaluation/results/retrieval_questions.csv and retrieval_events.csv")

event    class  may2024_docs_allowed   physics  impacts   indices   quality                      all_docs
 E216    super                  True 13 (0.78) 2 (0.80) 13 (0.76) 21 (0.85) [2, 3, 7, 11, 12, 13, 20, 21]
 E158  intense                 False  3 (0.80) 2 (0.81) 20 (0.75) 21 (0.85)          [1, 2, 3, 7, 20, 21]
 E231  intense                 False  3 (0.80) 2 (0.81) 20 (0.75) 21 (0.85)          [1, 2, 3, 7, 20, 21]
 E058  intense                 False  3 (0.80) 2 (0.80) 20 (0.75)         -              [1, 2, 3, 7, 20]
 E027  intense                 False  3 (0.80) 2 (0.80) 20 (0.75)         -              [1, 2, 3, 7, 20]
 E256  intense                 False  3 (0.80) 2 (0.80) 20 (0.75) 21 (0.85)          [1, 2, 3, 7, 20, 21]
 E039 moderate                 False  3 (0.79) 2 (0.81) 20 (0.77)         -         [2, 3, 7, 11, 20, 21]
 E117 moderate                 False  3 (0.79) 2 (0.81) 20 (0.77) 21 (0.85)         [2, 3, 7, 11, 20, 21]
 E095 moderate                 False  3 (0.79)

### 05 knowledge base and retrieval findings

- Corpus: 21 downloaded documents (7.3 MB, retrieved 2026-10-25) with a manifest of source, role and licence status; 20 indexed (the Kyoto service home page is excluded). Sources: 14 NOAA SWPC documents, GFZ (Kp), Kyoto WDC (Dst description and version definitions), a NASA tutorial, Hayakawa et al. 2025 (CC BY 4.0) on May 2024, and the Wikipedia May 2024 summary (CC BY-SA 4.0). Licences of the Kyoto and NASA documents are unverified.
- Extraction: 198,868 characters; rules per source type (NOAA `<article>`, Wikipedia content area, PDFs via pypdf; the paper cut before its ORCID list, Wikipedia aurora location lists dropped).
- Chunks: 290 (mean 715 characters; 900-character target, 120 overlap). The Hayakawa paper gives 122 chunks and the two May 2024 documents 137 of the 290. Cosmetic flaw: some chunks start with a stray ". ".
- Index: BAAI/bge-small-en-v1.5 (384 dimensions), Chroma, cosine similarity, 290 chunks.
- Retrieval check, 18 hand-labelled questions: hit@1 0.78, hit@3 1.0, hit@5 1.0, MRR 0.889. As first labelled: 0.72 / 0.94 / 0.94 / 0.833; one label (spacecraft charging) was corrected after reading the text, so the corrected figures are not independent. The set is small and still needs a proper review before I quote it.
- Similarity cannot separate supported from unsupported questions on related topics (unsupported 0.47-0.79; real questions' top-1 0.749-0.854). "No supporting source" must come from a claim-level check in notebook 06.
- The May 2024 documents are returned only for events whose window overlaps 2024-05-08 to 2024-05-20: 0 of 14 other evaluation events received them.
- One query per event gives near-identical results across events; several targeted queries (physics, impacts by storm level, index meaning, provisional-versus-final Dst) work better. Provisional-Dst events always retrieve Kyoto's definition of the provisional index (similarity 0.85).
- Retrieved background is class-level, not event-level, outside May 2024. No official NOAA post-event assessment of May 2024 was found; no NOAA pages exist for pipeline or aviation impacts.